# Step 04.1 — Linear：从矩阵乘法到可学习层

这一节只建立一个核心认识：

> `nn.Linear` 的数学核心仍然只是你已经会的矩阵乘法。

区别在于：其中的矩阵 `W` 不再由我们手工指定，而会在训练中被模型逐渐学出来。

## 1. 先不用神经网络 API，手算一次线性变换

输入向量有 2 个特征：

`x = [2, -1]`

我们希望输出 3 个新特征，所以准备 3 组权重，每组都要读取输入的 2 个特征。

In [ ]:
import mlx.core as mx

x = mx.array([2.0, -1.0])

W = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

print("x shape:", x.shape)
print("W shape:", W.shape)

`W.shape = (3, 2)`：

- 3 行：我们想产生 3 个输出特征；
- 每行 2 个权重：每个输出都要读取输入的 2 个特征。

第一行 `[1, 0]` 产生第 1 个输出；第二行 `[0, 1]` 产生第 2 个输出；第三行 `[1, 1]` 产生第 3 个输出。

In [ ]:
y = x @ W.T

print("W.T shape:", W.T.shape)
print("y:", y)
print("y shape:", y.shape)

手算：

- 第 1 个输出：`2×1 + (-1)×0 = 2`
- 第 2 个输出：`2×0 + (-1)×1 = -1`
- 第 3 个输出：`2×1 + (-1)×1 = 1`

所以：

`[2, -1] -> [2, -1, 1]`

这就是一次 `2维 -> 3维` 的 Linear transformation。

## 2. Linear 的“特征混合”是什么意思？

第三个输出不是复制输入，而是：

`y₂ = 1×x₀ + 1×x₁`

所以一个输出维度可以是多个输入维度的**加权组合**。

真实神经网络中的权重不会是这些简单的 0 和 1，而是大量浮点数。训练的任务之一，就是找到有用的这些权重。

## 3. 一次处理多个向量

神经网络通常不会只处理一个 `x`。假设一次有两个输入：

In [ ]:
X = mx.array([
    [2.0, -1.0],
    [3.0,  4.0],
])

Y = X @ W.T

print("X shape:", X.shape)
print("W.T shape:", W.T.shape)
print("Y:")
print(Y)
print("Y shape:", Y.shape)

shape 流转：

`(2, 2) @ (2, 3) -> (2, 3)`

两个输入向量都使用**同一套 W**。

以后输入变成 `(B, T, C)` 时，Linear 仍然主要作用在最后那个 `C` 维度上；前面的 Batch 和 Time 可以一起保留下来。

## 4. MLX 的 `nn.Linear`

`mlx.nn` 是 MLX 提供的 neural-network module library。

`nn.Linear(2, 3)` 的意思是：输入最后一维长度为 2，输出最后一维长度为 3。

In [ ]:
import mlx.nn as nn

layer = nn.Linear(2, 3, bias=False)

print("Linear weight shape:", layer.weight.shape)
print("input shape :", X.shape)

out = layer(X)
mx.eval(out)

print("output shape:", out.shape)

MLX 中这个层的 `weight.shape` 是 `(3, 2)`。

它的核心仍然等价于：

`X @ weight.T`

现在权重是随机初始化的，所以具体输出数字暂时没有意义。后面训练时，这些权重才会根据 loss 和 gradient 被修改。

### 本节结论

- Linear 的数学核心：矩阵乘法；
- `2 -> 3` 表示把 2 个输入特征重新组合成 3 个输出特征；
- 每个输出都是所有输入的一个加权组合；
- `nn.Linear` 只是把这个矩阵包装成神经网络中可管理、之后可学习的参数层。
